# Unit 04 - Metrics (Exercise)

**Atoms:** `U04-A5`, `U04-A6`, `U04-A7` · **Runtime:** ~20 seconds · **Goal:** compute funnel lifts, two `OEC`s, and two `CTR` definitions.

## Without code

Targets with seed 42: checkout winner = control; revenue-heavy `OEC` favours treatment while conversion-heavy `OEC` favours control; user-level `CTR` gap **larger** than session-level gap, because treatment users have more sessions and the extra sessions dilute the per-session rate.

## 1. The question

Decide winners under three different metric regimes.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

In [ ]:
n = 2000
D = np.repeat([0, 1], n // 2)
view = np.ones(n)
browse = np.random.binomial(1, np.where(D==1, 0.72, 0.70))
cart = np.random.binomial(1, np.where(D==1, 0.40, 0.32) * browse)
checkout = np.random.binomial(1, np.where(D==1, 0.45, 0.62) * cart)
order_value = np.where(D==1, np.random.lognormal(mean=3.8, sigma=0.35, size=n),
                              np.random.lognormal(mean=3.4, sigma=0.35, size=n))
revenue = checkout * order_value
funnel = pd.DataFrame({'D_i': D, 'browse': browse, 'cart': cart, 'checkout': checkout, 'revenue': revenue})
arm = funnel.groupby('D_i').agg(
    browse_rate=('browse','mean'), checkout_rate=('checkout','mean'), avg_revenue=('revenue','mean'))
# Each metric divided by the best arm on that metric, so no metric dominates by unit alone
scaled = arm / arm.max()
print(arm.round(4))

## 4. TODO - funnel lift

Compute relative lift `(treatment - control) / control` at the **checkout** stage. Print a float. Expect **negative** lift (~ -0.05 to -0.10).

In [ ]:
# TODO
checkout_lift = None
assert checkout_lift is not None
print('Checkout relative lift:', round(checkout_lift, 4))
assert checkout_lift < 0

## 5. TODO - OEC weights

Complete `oec_score` and `winner` using the `scaled` table from above (same logic as the demo). Print both winners. The two weightings should disagree - that is the whole point.

In [ ]:
weights_biz = {'browse_rate': 0.1, 'checkout_rate': 0.2, 'avg_revenue': 0.7}
weights_conv = {'browse_rate': 0.1, 'checkout_rate': 0.7, 'avg_revenue': 0.2}

def oec_score(arm_id, w):
    # TODO: weighted sum over the scaled metrics for this arm
    return None

def winner(w):
    # TODO: return 'treatment' or 'control'
    return None

print('Revenue-heavy winner:', winner(weights_biz))
print('Conversion-heavy winner:', winner(weights_conv))
assert winner(weights_biz) == 'treatment'
assert winner(weights_conv) == 'control'

## 6. TODO - user vs session CTR

Compute the treatment-minus-control `CTR` gap two ways: clicks per **user** and clicks per **session**. Treatment users visit more often, so the two denominators do not tell the same story. Assert that the user-level gap is the larger of the two.

In [ ]:
users = pd.DataFrame({'user_id': np.arange(500), 'D_i': np.random.binomial(1, 0.5, 500)})
users['sessions'] = np.where(users.D_i==1, np.random.poisson(2.2, 500), np.random.poisson(1.5, 500))
users['clicks'] = np.random.binomial(users['sessions'], np.where(users.D_i==1, 0.35, 0.30))
# TODO
user_gap = None
session_gap = None
assert user_gap is not None and session_gap is not None
print('CTR gap user-level:', round(user_gap,4), 'session-level:', round(session_gap,4))
assert user_gap > session_gap

**Takeaway:** Define the metric before you analyse. **Unit:** [V1](../V1/units/unit-04-hypothesis-and-metrics/README.md) · [V2](../V2/units/unit-04-hypothesis-and-metrics/README.md)

## Hints
- Relative lift at checkout: group means then `(t-c)/c`.
- Normalise each metric to 0-1 across arms before weighting.
- `CTR = total clicks / total users` or `/ total sessions`.

## Spoiler
```python
checkout_by_arm = funnel.groupby('D_i')['checkout'].mean()
checkout_lift = (checkout_by_arm[1] - checkout_by_arm[0]) / checkout_by_arm[0]

def oec_score(arm_id, w):
    row = scaled.loc[arm_id]
    return sum(w[k] * row[k] for k in w)

def winner(w):
    return 'treatment' if oec_score(1, w) > oec_score(0, w) else 'control'

totals = users.groupby('D_i')[['clicks', 'sessions']].sum()
n_users = users.groupby('D_i')['user_id'].count()
ctr_user = totals['clicks'] / n_users
ctr_session = totals['clicks'] / totals['sessions']
user_gap = ctr_user[1] - ctr_user[0]
session_gap = ctr_session[1] - ctr_session[0]
```